In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/kimanaru/dataset-tien-xu-li/val_ml_clean_text.csv
/kaggle/input/datasets/kimanaru/dataset-tien-xu-li/train_ml_clean_text.csv
/kaggle/input/datasets/kimanaru/dataset-tien-xu-li/test_ml_clean_text.csv


In [2]:
import pandas as pd
from pathlib import Path

RESULT_DIR = Path("/kaggle/input/datasets/kimanaru/dataset-tien-xu-li")

train_df = pd.read_csv(RESULT_DIR / "train_ml_clean_text.csv")
val_df = pd.read_csv(RESULT_DIR / "val_ml_clean_text.csv")
test_df = pd.read_csv(RESULT_DIR / "test_ml_clean_text.csv")

train_df["ml_clean_text"] = train_df["ml_clean_text"].fillna("").astype(str)
val_df["ml_clean_text"] = val_df["ml_clean_text"].fillna("").astype(str)
test_df["ml_clean_text"] = test_df["ml_clean_text"].fillna("").astype(str)

train_df["category"] = train_df["category"].astype(str)
val_df["category"] = val_df["category"].astype(str)
test_df["category"] = test_df["category"].astype(str)

X_train = train_df["ml_clean_text"]
y_train = train_df["category"]

X_val = val_df["ml_clean_text"]
y_val = val_df["category"]

X_test = test_df["ml_clean_text"]
y_test = test_df["category"]

# Thử nghiệm 

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBClassifier
import numpy as np
import pandas as pd
import time

# =========================
# Chuẩn hóa dữ liệu đầu vào
# =========================
X_train_text = X_train.fillna("").astype(str)
X_val_text = X_val.fillna("").astype(str)
X_test_text = X_test.fillna("").astype(str)

y_train_text = y_train.astype(str)
y_val_text = y_val.astype(str)
y_test_text = y_test.astype(str)

# =========================
# Encode nhãn cho XGBoost multi-class
# =========================
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train_text)
y_val_enc = label_encoder.transform(y_val_text)
y_test_enc = label_encoder.transform(y_test_text)

num_classes = len(label_encoder.classes_)
print("Number of classes:", num_classes)
print("Classes:", list(label_encoder.classes_))

# Cân bằng lớp để giảm thiên lệch về các lớp có nhiều mẫu
sample_weight_train = compute_sample_weight(
    class_weight="balanced",
    y=y_train_enc
)


Number of classes: 15
Classes: ['BLACK VOICES', 'BUSINESS', 'COMEDY', 'ENTERTAINMENT', 'FOOD & DRINK', 'HEALTHY LIVING', 'HOME & LIVING', 'PARENTING', 'PARENTS', 'POLITICS', 'QUEER VOICES', 'SPORTS', 'STYLE & BEAUTY', 'TRAVEL', 'WELLNESS']


In [4]:
!pip install sentence-transformers catboost

In [5]:
from sentence_transformers import SentenceTransformer

sbert_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

X_train_emb = sbert_model.encode(
    X_train_text,
    batch_size=64,
    show_progress_bar=True
)
X_val_emb = sbert_model.encode(
    X_val_text,
    batch_size=64,
    show_progress_bar=True
)
X_test_emb = sbert_model.encode(
    X_test_text,
    batch_size=64,
    show_progress_bar=True
)



modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1621 [00:00<?, ?it/s]

Batches:   0%|          | 0/348 [00:00<?, ?it/s]

Batches:   0%|          | 0/348 [00:00<?, ?it/s]

In [6]:
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, classification_report
import pandas as pd
import time


# Danh sách hyperparameter cần thử
param_list = [
    {
        "depth": 4,
        "learning_rate": 0.05,
        "l2_leaf_reg": 3
    },
    {
        "depth": 6,
        "learning_rate": 0.05,
        "l2_leaf_reg": 3
    },
    {
        "depth": 6,
        "learning_rate": 0.03,
        "l2_leaf_reg": 5
    },
    {
        "depth": 8,
        "learning_rate": 0.03,
        "l2_leaf_reg": 5
    },
    {
        "depth": 6,
        "learning_rate": 0.01,
        "l2_leaf_reg": 10
    }
]

results = []
best_model = None
best_f1 = -1

for i, params in enumerate(param_list):
    print(f"\n===== Thử nghiệm {i+1}/{len(param_list)} =====")
    print(params)

    start_time = time.time()

    model = CatBoostClassifier(
        iterations=500,
        learning_rate=params["learning_rate"],
        depth=params["depth"],
        l2_leaf_reg=params["l2_leaf_reg"],

        loss_function="MultiClass",
        eval_metric="TotalF1",

        auto_class_weights="Balanced",

        early_stopping_rounds=50,
        random_seed=42,
        verbose=100
    )

    model.fit(
        X_train_emb,
        y_train,
        eval_set=(X_val_emb, y_val),
        use_best_model=True
    )

    y_val_pred = model.predict(X_val_emb)

    val_macro_f1 = f1_score(
        y_val,
        y_val_pred,
        average="macro"
    )

    elapsed_time = time.time() - start_time

    results.append({
        "depth": params["depth"],
        "learning_rate": params["learning_rate"],
        "l2_leaf_reg": params["l2_leaf_reg"],
        "val_macro_f1": val_macro_f1,
        "best_iteration": model.get_best_iteration(),
        "time_seconds": elapsed_time
    })

    if val_macro_f1 > best_f1:
        best_f1 = val_macro_f1
        best_model = model

results_df = pd.DataFrame(results)
results_df.sort_values("val_macro_f1", ascending=False)


===== Thử nghiệm 1/5 =====
{'depth': 4, 'learning_rate': 0.05, 'l2_leaf_reg': 3}
0:	learn: 0.1251162	test: 0.1252698	best: 0.1252698 (0)	total: 1.05s	remaining: 8m 43s
100:	learn: 0.5817500	test: 0.5767639	best: 0.5767639 (100)	total: 1m 44s	remaining: 6m 51s
200:	learn: 0.6222686	test: 0.6136112	best: 0.6140126 (199)	total: 3m 22s	remaining: 5m 1s
300:	learn: 0.6428414	test: 0.6303440	best: 0.6308211 (298)	total: 4m 57s	remaining: 3m 17s
400:	learn: 0.6574492	test: 0.6416752	best: 0.6418887 (392)	total: 6m 29s	remaining: 1m 36s
499:	learn: 0.6675849	test: 0.6486563	best: 0.6488319 (479)	total: 8m	remaining: 0us

bestTest = 0.648831852
bestIteration = 479

Shrink model to first 480 iterations.

===== Thử nghiệm 2/5 =====
{'depth': 6, 'learning_rate': 0.05, 'l2_leaf_reg': 3}
0:	learn: 0.1938143	test: 0.1958809	best: 0.1958809 (0)	total: 3.92s	remaining: 32m 37s
100:	learn: 0.6256238	test: 0.6033110	best: 0.6037004 (98)	total: 6m 35s	remaining: 26m 2s
200:	learn: 0.6643728	test: 0.63130

,depth,learning_rate,l2_leaf_reg,val_macro_f1,best_iteration,time_seconds
1,6,0.05,3,0.633336,482,1900.530643
3,8,0.03,5,0.630878,489,3739.206459
2,6,0.03,5,0.620673,499,1901.800241
0,4,0.05,3,0.619164,479,483.272013
4,6,0.01,10,0.581296,496,1868.237814


In [7]:
import joblib

# lưu kết quả tạm
results_df = pd.DataFrame(results)
results_df.to_csv("/kaggle/working/catboost_results.csv", index=False)

# lưu best model
joblib.dump(best_model, "/kaggle/working/best_catboost.pkl")

['/kaggle/working/best_catboost.pkl']

In [8]:
y_test_pred = best_model.predict(X_test_emb)

test_macro_f1 = f1_score(
    y_test,
    y_test_pred,
    average="macro"
)

print("Best Validation Macro F1:", best_f1)
print("Test Macro F1:", test_macro_f1)

print(
    classification_report(
        y_test,
        y_test_pred
    )
)

Best Validation Macro F1: 0.6333363371018774
Test Macro F1: 0.6350798741016318
                precision    recall  f1-score   support

  BLACK VOICES       0.41      0.57      0.48       687
      BUSINESS       0.47      0.69      0.56       899
        COMEDY       0.38      0.53      0.44       810
 ENTERTAINMENT       0.75      0.63      0.69      2605
  FOOD & DRINK       0.71      0.84      0.77       951
HEALTHY LIVING       0.45      0.39      0.42      1004
 HOME & LIVING       0.60      0.72      0.66       648
     PARENTING       0.59      0.64      0.61      1319
       PARENTS       0.40      0.44      0.42       593
      POLITICS       0.91      0.75      0.82      5341
  QUEER VOICES       0.67      0.70      0.68       952
        SPORTS       0.68      0.82      0.75       761
STYLE & BEAUTY       0.77      0.78      0.78      1472
        TRAVEL       0.75      0.76      0.76      1485
      WELLNESS       0.74      0.67      0.70      2692

      accuracy         